# 05 — Segment and Fenwick Trees

## Why This Matters for DSA
Linear array queries are subject to a strict trade-off: a raw array allows point updates in $O(1)$ time but requires $O(n)$ time for range queries (e.g. Range Sum Query). A prefix sum array (`02_Sliding_Window_and_Prefix_Sum`) reverses this trade-off, enabling range sum queries in $O(1)$ time at the expense of $O(n)$ point updates. When the array is dynamic and receives both queries and updates, neither linear structure is acceptable. **Segment Trees** and **Fenwick Trees (Binary Indexed Trees)** bridge this gap. By representing array intervals hierarchically, they balance both operations to run in $O(\log n)$ time. These structures are the foundation for advanced range query processing, interval manipulation, and dynamic prefix-based logic in competitive programming and large-scale database indices.

## Prerequisites
- `01_Complexity_Analysis` (Phase 01) — recursion math and logarithmic complexity bounds
- `02_Recursion` (Phase 01) — divide-and-conquer recursive patterns
- `02_Sliding_Window_and_Prefix_Sum` (Phase 02) — understanding static prefix sum arrays and range sum logic

## Learning Objectives
By the end of this notebook, you will be able to:
- Explain the point update vs. range query trade-offs between static arrays, prefix sums, Segment Trees, and Fenwick Trees.
- Build a Segment Tree recursively in $O(n)$ time and query/update intervals in $O(\log n)$ time.
- Mathematically prove why a Segment Tree requires a flat array representation of size $4n$.
- Implement a **Fenwick Tree (Binary Indexed Tree)** using 1-based indexing and bitwise operators (`idx & -idx`).
- Compare Segment and Fenwick Trees on flexibility, space constraints, constant-factor speed, and type of query operators.
- Explain the core concept of **Lazy Propagation** for performing range updates in $O(\log n)$ time.

## Checklist Before Moving On
- [ ] I can state the range query and point update complexity for all four representations (Array, Prefix Sum, Segment Tree, BIT)
- [ ] I can write the recursive build, update, and query functions for a Segment Tree from memory
- [ ] I can explain why $4n$ array slots are sufficient to store a tree representing an array of size $n$
- [ ] I can explain what `idx & (-idx)` computes in two's complement and write its output for a given index
- [ ] I can write the update and prefix query loops of a Fenwick Tree from memory
- [ ] I can choose between Segment Trees and Fenwick Trees based on operator properties and performance requirements
- [ ] I can explain how Lazy Propagation avoids updating leaf nodes immediately

## Table of Contents
1. The Range Query & Point Update Dilemma
2. Segment Tree Layout and Array Index Math
3. Segment Tree Operations — Build, Query, and Point Update
4. Fenwick Tree (Binary Indexed Tree) Bitwise Fundamentals
5. Fenwick Tree Operations — Update and Prefix Query
6. Segment Tree vs. Fenwick Tree Comparison Matrix
7. Advanced Topic — Lazy Propagation Preview
8. Decision Guide — Prefix Sum vs. Fenwick vs. Segment Tree
9. Phase Checkpoint, Cheat Sheet, and Answer Key
10. LeetCode Practice Problems


## 1. The Range Query & Point Update Dilemma

### The Why
Many practical workloads involve dynamic arrays that receive interleaving queries and modifications. For instance, in database indexing or network packet analysis, we must frequently query range stats (e.g. total volume in a time range) while updating point entries. Classic data structures fail this test: static structures are either too slow to update ($O(n)$ prefix sums) or too slow to query ($O(n)$ arrays). Segment Trees and Fenwick Trees resolve this dilemma by using hierarchical interval divisions, balancing both operations at $O(\log n)$ time.

### Core Comparison Table

| Structure | Range Query (e.g. Sum) | Point Update | Space Complexity | Best Use Case |
|---|---|---|---|---|
| **Naive Array** | $O(n)$ | $O(1)$ | $O(n)$ | Read-sparse, write-heavy workloads |
| **Prefix Sum Array** | $O(1)$ | $O(n)$ | $O(n)$ | Static arrays (zero updates) |
| **Segment Tree** | $O(\log n)$ | $O(\log n)$ | $O(n)$ (actually $4n$) | Dynamic arrays, general associative operators (min, max, gcd) |
| **Fenwick Tree (BIT)** | $O(\log n)$ | $O(\log n)$ | $O(n)$ (compact) | Dynamic arrays, prefix-invertible operators (sum, product) |

### Common Pitfalls
- **Using Prefix Sums for Dynamic Arrays:** Rebuilding a static prefix sum array after every single update. If an array receives $q$ updates, this yields $O(q n)$ time, which is worse than a simple array ($O(q)$ updates) and much worse than segment/Fenwick trees ($O(q \log n)$).


## 2. Segment Tree Layout and Array Index Math

### The Why
A Segment Tree represents an array by dividing it into intervals. The root represents the entire array range $[0..n-1]$. Its left child represents the left half $[0..mid]$, and its right child represents the right half $[mid+1..n-1]$, continuing recursively down to leaf nodes representing single elements $[i..i]$. Like binary heaps (`01_Heaps_and_Priority_Queues`), we store this tree in a flat array, mapping parent and child nodes using bitwise index calculations. Understanding this index mapping and the memory constraints is the prerequisite to implementing Segment Trees.

### Core Mechanism
- **Array Index Math (1-based tree array indexing):**
  For a node at index `node`:
  - **Left Child Index:** `2 * node` (implement as `node << 1`)
  - **Right Child Index:** `2 * node + 1` (implement as `(node << 1) | 1`)
- **Why $4n$ Size is Sufficient:**
  An array of size $n$ has $n$ leaf nodes. If $n$ is a power of 2, the segment tree is a full binary tree with height $\log_2 n$, containing exactly $2n-1$ nodes.
  - However, if $n$ is not a power of 2, the height increases to $\lceil \log_2 n \rceil$, and the bottom level will contain empty slots.
  - In the worst case (e.g. $n = 2^k + 1$), the height is $k+1$. The total number of nodes at levels $0$ through $k$ is $2^{k+1}-1$, and the bottom level can have indices extending up to $2^{k+2}-2$.
  - Since $2^{k+2} \approx 4n$, allocating an array of size **$4n$** guarantees we will never dereference an index out of bounds, regardless of whether $n$ is a power of 2.

### Common Pitfalls
- **Allocating $2n$ or $2^{\lceil \log_2 n \rceil + 1}$ without bounds checks:** Allocating exactly $2n$ space and crashing. Always allocate $4n$ space for the tree array to avoid out-of-bounds errors on general array sizes $n$.


## 3. Segment Tree Operations — Build, Query, and Point Update

### The Why
Segment Tree operations use recursive divide-and-conquer. We build the tree bottom-up, query ranges by dividing them into segments, and update points by traversing down to the leaf node and rebuilding path values back to the root. Seeing this code compiles and runs is key to understanding the three cases of range queries.

### Core Mechanism
- **Build Operation ($O(n)$):** Divide the range $[start..end]$ in half, build the left subtree, build the right subtree, and then merge child results at the current node.
- **Point Update ($O(\log n)$):** Traverse down to the leaf node corresponding to the target index, update its value, and then update all parent nodes along the recursion path on the way back up.
- **Range Query ($O(\log n)$):** A query for range $[L..R]$ checks the current node's range $[start..end]$ against three cases:
  1. **No Overlap:** $[L..R]$ is completely outside $[start..end]$. Return the identity element (e.g. $0$ for sum queries).
  2. **Complete Overlap:** $[start..end]$ is completely inside $[L..R]$ (i.e. $L \le start \le end \le R$). Return the current node's stored value.
  3. **Partial Overlap:** Range is partially inside and partially outside. Recurse on both children, merge their results, and return.


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

class SegmentTree {
private:
    vector<int> tree;
    int n;

    // Recursive build: O(n) time
    void build(const vector<int>& arr, int node, int start, int end) {
        if (start == end) {
            tree[node] = arr[start];
            return;
        }
        int mid = start + (end - start) / 2;
        build(arr, 2 * node, start, mid);       // Build left child
        build(arr, 2 * node + 1, mid + 1, end);  // Build right child
        
        // Merge step (Sum operator)
        tree[node] = tree[2 * node] + tree[2 * node + 1];
    }

    // Recursive update: O(log n) time
    void update(int node, int start, int end, int idx, int val) {
        if (start == end) {
            tree[node] = val;
            return;
        }
        int mid = start + (end - start) / 2;
        if (idx <= mid) {
            update(2 * node, start, mid, idx, val);
        } else {
            update(2 * node + 1, mid + 1, end, idx, val);
        }
        // Update parent after child is modified
        tree[node] = tree[2 * node] + tree[2 * node + 1];
    }

    // Recursive query: O(log n) time
    int query(int node, int start, int end, int l, int r) {
        // Case 1: No Overlap
        if (r < start || end < l) {
            return 0; // Identity element for addition
        }
        // Case 2: Complete Overlap
        if (l <= start && end <= r) {
            return tree[node];
        }
        // Case 3: Partial Overlap
        int mid = start + (end - start) / 2;
        int leftSum = query(2 * node, start, mid, l, r);
        int rightSum = query(2 * node + 1, mid + 1, end, l, r);
        return leftSum + rightSum;
    }

public:
    SegmentTree(const vector<int>& arr) {
        n = arr.size();
        tree.resize(4 * n, 0); // ALLOCATE 4n SIZE
        if (n > 0) build(arr, 1, 0, n - 1);
    }

    void update(int idx, int val) {
        update(1, 0, n - 1, idx, val);
    }

    int query(int l, int r) {
        return query(1, 0, n - 1, l, r);
    }
};

int main() {
    vector<int> arr = {1, 3, 5, 7, 9, 11};
    SegmentTree segTree(arr);

    cout << "Range Sum Query (1, 3): " << segTree.query(1, 3) << "\n"; // Expected: 3 + 5 + 7 = 15
    
    cout << "Updating index 2 to value 10 (was 5)...\n";
    segTree.update(2, 10);
    
    cout << "Range Sum Query (1, 3): " << segTree.query(1, 3) << "\n"; // Expected: 3 + 10 + 7 = 20
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 4. Fenwick Tree (Binary Indexed Tree) Bitwise Fundamentals

### The Why
Segment Trees require $4n$ array space and recursive function calls, which introduces overhead. Peter Fenwick designed the **Fenwick Tree (or Binary Indexed Tree)** to solve dynamic prefix sums using a compact array of size $n+1$ and extremely fast bitwise traversal. The Fenwick Tree is a masterclass in utilizing integer binary representation to define interval ranges.

### Core Mechanism
- **Bitwise Interval Partitioning:**
  In a Fenwick Tree, the value at index `idx` (using 1-based indexing) is the sum of the array range $[(idx - LSB(idx) + 1)..idx]$, where $LSB(idx)$ represents the **Least Significant Set Bit** of `idx` in binary.
  - For example, if $idx = 12$ ($1100_2$):
    - $LSB(12) = 4$ ($0100_2$).
    - Node 12 stores the sum of the range $[12 - 4 + 1..12] = [9..12]$.
- **Bitwise Formula for LSB:**
  $$LSB(x) = x \,\&\, (-x)$$
  - In two's complement, negation flips all bits of $x$ and adds 1. This keeps only the lowest set bit identical, so ANDing them isolates exactly the LSB.
  - Try it: $12 = 00001100_2$. $-12 = 11110100_2$. $12 \,\&\, (-12) = 00000100_2 = 4$.
- **Index Traversal Loops:**
  - **Query (Prefix Sum 1..idx):** We sum the values of tree cells and move *down* by subtracting the LSB. Loop step: `idx -= idx & (-idx)`.
  - **Update (Add delta to idx):** We add `delta` to tree cells and move *up* to update parent intervals by adding the LSB. Loop step: `idx += idx & (-idx)`.

```
Index binary representation transitions (e.g. Query starting at 7):
  7 (0111) -> subtract LSB (1) -> 6 (0110) -> subtract LSB (2) -> 4 (0100) -> subtract LSB (4) -> 0 (Done)
  PrefixSum(7) = BIT[7] + BIT[6] + BIT[4]
```

### Common Pitfalls
- **Using 0-Based Indexing:** Passing `idx = 0` to a Fenwick Tree. If `idx = 0`, `0 & (-0) = 0`. The update or query step `idx += 0` results in an infinite loop. **Fenwick Trees must use 1-based indexing.** Always map 0-based array index `i` to 1-based tree index `i + 1`.


## 5. Fenwick Tree Operations — Update and Prefix Query

### The Why
Fenwick Tree operations are entirely iterative, requiring no recursion. This results in tiny code and incredibly fast execution speeds. This section demonstrates how to implement point updates, prefix sum queries, and range sum queries on a Fenwick Tree.

### Core Mechanism
- **Update `update(idx, delta)` ($O(\log n)$):** Starting at index `idx` (1-based), add `delta` to `BIT[idx]`, and advance to the next parent index using `idx += idx & (-idx)`. Repeat until `idx > n`.
- **Prefix Query `query(idx)` ($O(\log n)$):** Starting at index `idx` (1-based), add `BIT[idx]` to a running sum, and move down to the next child index using `idx -= idx & (-idx)`. Repeat until `idx == 0`.
- **Range Sum Query `rangeQuery(L, R)` ($O(\log n)$):** Calculate prefix sum up to $R$ and subtract the prefix sum up to $L-1$:
  $$RangeSum(L, R) = PrefixSum(R) - PrefixSum(L-1)$$


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
using namespace std;

class FenwickTree {
private:
    vector<int> BIT;
    int n;

public:
    // Initialize tree with size n
    FenwickTree(int size) : n(size) {
        BIT.assign(n + 1, 0); // 1-based index needs n+1 space
    }

    // Construct Fenwick Tree from array: O(n log n) or O(n) build
    FenwickTree(const vector<int>& arr) : n(arr.size()) {
        BIT.assign(n + 1, 0);
        for (int i = 0; i < n; i++) {
            update(i + 1, arr[i]); // Map 0-based index to 1-based index
        }
    }

    // Point Update: Add delta to index idx (1-based) in O(log n)
    void update(int idx, int delta) {
        for (; idx <= n; idx += idx & (-idx)) {
            BIT[idx] += delta;
        }
    }

    // Prefix Query: Return sum from 1 to idx (1-based) in O(log n)
    int query(int idx) {
        int sum = 0;
        for (; idx > 0; idx -= idx & (-idx)) {
            sum += BIT[idx];
        }
        return sum;
    }

    // Range Sum Query: Return sum from l to r (1-based) in O(log n)
    int rangeQuery(int l, int r) {
        return query(r) - query(l - 1);
    }
};

int main() {
    vector<int> arr = {1, 3, 5, 7, 9, 11};
    FenwickTree bit(arr);

    // Sum of range [2..4] (1-based) -> elements arr[1]+arr[2]+arr[3] = 3+5+7 = 15
    cout << "Range Sum Query (2, 4): " << bit.rangeQuery(2, 4) << "\n"; // Expected: 15
    
    cout << "Adding 5 to index 3 (element becomes 5 + 5 = 10)...\n";
    bit.update(3, 5);
    
    cout << "Range Sum Query (2, 4): " << bit.rangeQuery(2, 4) << "\n"; // Expected: 3 + 10 + 7 = 20
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 6. Segment Tree vs. Fenwick Tree Comparison Matrix

### The Why
Both Segment Trees and Fenwick Trees offer identical asymptotic complexities ($O(\log n)$ range query, $O(\log n)$ point update). However, they differ in practical implementation, flexibility, and memory overhead. Choosing the right one is a core engineering decision.

### Complexity & Property Comparison Matrix

| Property | Segment Tree | Fenwick Tree (BIT) | Winner |
|---|---|---|---|
| **Memory Space** | $4n$ array slots | $n + 1$ array slots | **Fenwick Tree** (uses 4x less memory) |
| **Implementation** | Recursive, verbose | Iterative, extremely concise | **Fenwick Tree** (easier to write) |
| **Constant Factors** | Larger (recursion overhead) | Small (fast bitwise operations) | **Fenwick Tree** (faster in practice) |
| **Operator Flexibility** | Any associative operator (Sum, Min, Max, GCD) | Prefix-invertible/friendly operators (Sum, Product, XOR) | **Segment Tree** (much more general) |
| **Lazy Propagation** | Supported (range updates in $O(\log n)$) | Extremely difficult / limited | **Segment Tree** |

### Key Takeaway
- If you only need range sum/product/XOR queries with point updates, **Fenwick Tree** is the optimal choice due to speed and memory efficiency.
- If you need range min/max queries or range updates (lazy propagation), **Segment Tree** is required.


## 7. Advanced Topic — Lazy Propagation Preview

### The Why
A standard Segment Tree point update takes $O(\log n)$ time. If we need to update a *range* of elements $[L..R]$ (e.g. add value $v$ to all elements in the range), updating each element individually would take $O(n \log n)$ time. **Lazy Propagation** is an optimization technique that reduces range updates to $O(\log n)$ time. It works by delaying updates to child nodes until those nodes are actually visited.

### Core Mechanism
- **The Lazy Array:** We maintain a secondary array `lazy[4n]`, initialized to 0, which stores pending updates for subtrees.
- **On Range Update `updateRange(L, R, val)`:**
  - If the current node's range $[start..end]$ is completely inside $[L..R]$, we apply the update to the current node (e.g. `tree[node] += (end - start + 1) * val` for sum) and store the pending update in `lazy[node]`. **We do not recurse into its children immediately.** This is the "lazy" step.
  - If the range is partially overlapping, we push the current node's lazy value down to its children, clear `lazy[node]`, recurse on both children, and merge their values.
- **On Query `query(L, R)`:**
  - Before reading a node's value, check if `lazy[node] != 0`. If it is, apply the pending update, push it down to children, clear `lazy[node]`, and then proceed with the query.
- **Result:** Range updates and range queries both run in $O(\log n)$ time because we only traverse down the tree paths necessary to fulfill the query, delaying updates to unvisited subtrees.


## 8. Decision Guide — Prefix Sum vs. Fenwick vs. Segment Tree

### The Why
Choosing the right interval query structure requires analyzing the operation mix and operators. This decision guide summarizes the rules of thumb.

### Decision Rules
- **Use Prefix Sum Arrays when:**
  - The array is static (no updates occur). Prefix sums give $O(1)$ range queries with zero overhead.
- **Use Fenwick Trees (BIT) when:**
  - The array is dynamic (frequent point updates), and the range operator is invertible/associative (Sum, XOR).
  - Memory footprint is constrained (Fenwick uses $n+1$ space vs. $4n$ for Segment Trees).
  - You want maximum execution speed (constant-factor performance is critical).
- **Use Segment Trees when:**
  - The operator is non-invertible (e.g. Min, Max, GCD). Fenwick trees cannot easily compute Range Minimum Queries (RMQ) under point updates.
  - You need to perform range updates (requiring Lazy Propagation).
  - The node structures are complex (e.g. storing multiple values like min, max, and count in each node).


## 9. Phase Checkpoint, Cheat Sheet, and Answer Key

### Range Query Data Structures Cheat Sheet

| Operation | Naive Array | Prefix Sum | Segment Tree | Fenwick Tree |
|---|---|---|---|---|
| **Point Update** | $O(1)$ | $O(n)$ | $O(\log n)$ | $O(\log n)$ |
| **Range Sum Query** | $O(n)$ | $O(1)$ | $O(\log n)$ | $O(\log n)$ |
| **Range Min/Max Query** | $O(n)$ | $O(n)$ | $O(\log n)$ | $O(n)$ (not easily supported) |
| **Range Update** | $O(n)$ | $O(n)$ | $O(\log n)$ (with lazy) | $O(n)$ (not easily supported) |
| **Space Complexity** | $n$ | $n$ | $4n$ | $n + 1$ |

### Checkpoint Questions
1. Explain why a Segment Tree array requires size $4n$ instead of $2n$ for an array of size $n$.
2. State the three overlap cases encountered during a Segment Tree range query.
3. What is the value of `10 & (-10)`? What does this value represent in the Fenwick Tree?
4. In a Fenwick Tree, why does `idx += idx & (-idx)` traverse up parent nodes during an update?
5. Why is a Fenwick Tree unable to easily handle Range Minimum Queries (RMQ) under point updates?
6. Explain how Lazy Propagation improves range updates from $O(n \log n)$ to $O(\log n)$ time.

### Answer Key
1. If $n$ is not a power of 2, the height is $\lceil \log_2 n \rceil$. The leaf nodes will reside on the bottom level, and empty slots in the full binary tree structure can push index values up to $4n$. Allocating $4n$ space prevents out-of-bounds errors.
2. The three overlap cases are: (1) No Overlap (return identity element, 0), (2) Complete Overlap (return current node's value), and (3) Partial Overlap (recurse on both children and merge).
3. $10 = 00001010_2$. $-10 = 11110110_2$. $10 \,\&\, (-10) = 00000010_2 = 2$. This value represents the Least Significant Set Bit (LSB) of 10, which indicates the length of the interval stored at index 10 (length 2).
4. Adding the LSB of `idx` advances the index to the next larger binary interval that includes `idx` as a subsegment, ensuring that all covering ranges are updated.
5. The minimum operator is not invertible. If we update a node value to be larger, we cannot deduce the new minimum of parent ranges without checking all other subsegments. Fenwick Trees rely on prefix operations, which require invertibility (like subtraction for sum).
6. Lazy Propagation delays updates to child subtrees. Instead of recursing to leaves immediately, it updates the current range node, stores the pending value in a `lazy[]` array, and propagates it downward only when a query or update visits those children later.

### Mini Project: Range Minimum Query (RMQ) Segment Tree
Implement a Segment Tree that supports Range Minimum Queries (RMQ) and point updates.
- **Strategy:** Modify the segment tree build, query, and update logic: replace the sum merge step `tree[node] = left + right` with `tree[node] = min(left, right)`. For the no-overlap query case, return infinity (`1e9`) instead of $0$ (since infinity is the identity element for the min operator).


In [ ]:
%%writefile temp.cpp
#include <iostream>
#include <vector>
#include <algorithm>
using namespace std;

class RangeMinSegmentTree {
private:
    vector<int> tree;
    int n;
    const int INF = 1e9; // Identity element for min operator

    void build(const vector<int>& arr, int node, int start, int end) {
        if (start == end) {
            tree[node] = arr[start];
            return;
        }
        int mid = start + (end - start) / 2;
        build(arr, 2 * node, start, mid);
        build(arr, 2 * node + 1, mid + 1, end);
        tree[node] = min(tree[2 * node], tree[2 * node + 1]);
    }

    void update(int node, int start, int end, int idx, int val) {
        if (start == end) {
            tree[node] = val;
            return;
        }
        int mid = start + (end - start) / 2;
        if (idx <= mid) {
            update(2 * node, start, mid, idx, val);
        } else {
            update(2 * node + 1, mid + 1, end, idx, val);
        }
        tree[node] = min(tree[2 * node], tree[2 * node + 1]);
    }

    int query(int node, int start, int end, int l, int r) {
        // Case 1: No Overlap
        if (r < start || end < l) {
            return INF; // Return infinity so it doesn't affect min()
        }
        // Case 2: Complete Overlap
        if (l <= start && end <= r) {
            return tree[node];
        }
        // Case 3: Partial Overlap
        int mid = start + (end - start) / 2;
        int leftMin = query(2 * node, start, mid, l, r);
        int rightMin = query(2 * node + 1, mid + 1, end, l, r);
        return min(leftMin, rightMin);
    }

public:
    RangeMinSegmentTree(const vector<int>& arr) {
        n = arr.size();
        tree.resize(4 * n, INF);
        if (n > 0) build(arr, 1, 0, n - 1);
    }

    void update(int idx, int val) {
        update(1, 0, n - 1, idx, val);
    }

    int query(int l, int r) {
        return query(1, 0, n - 1, l, r);
    }
};

int main() {
    vector<int> arr = {5, 2, 8, 1, 9, 3};
    RangeMinSegmentTree rmq(arr);

    cout << "Range Min Query (0, 4): " << rmq.query(0, 4) << "\n"; // Expected: 1 (at index 3)
    
    cout << "Updating index 3 to value 10 (was 1)...\n";
    rmq.update(3, 10);
    
    cout << "Range Min Query (0, 4): " << rmq.query(0, 4) << "\n"; // Expected: 2 (at index 1)
    return 0;
}


In [ ]:
!g++ -std=c++17 -O2 -Wall -o temp temp.cpp && ./temp

## 10. LeetCode Practice Problems

Use these problems to build fluency with interval query structures.

### Segment Tree Patterns

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 307 | [Range Sum Query - Mutable](https://leetcode.com/problems/range-sum-query-mutable/) | Medium | Section 3, directly — implement build, query, and point update |
| 303 | [Range Sum Query - Immutable](https://leetcode.com/problems/range-sum-query-immutable/) | Easy | Static array — implement using simple prefix sums (`02_Sliding_Window_and_Prefix_Sum`), no tree needed! |

### Fenwick Tree / Prefix Counting

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 315 | [Count of Smaller Numbers After Self](https://leetcode.com/problems/count-of-smaller-numbers-after-self/) | Hard | Process array from right to left. Use Fenwick Tree to record element frequencies, and query prefix sums up to `val - 1` |
| 493 | [Reverse Pairs](https://leetcode.com/problems/reverse-pairs/) | Hard | Similar to #315 — map elements to ranks, traverse right to left, and query prefix sums up to `val / 2.0` |
| 327 | [Count of Range Sum](https://leetcode.com/problems/count-of-range-sum/) | Hard | Count prefix sum intervals using coordinate compression and Fenwick Tree prefix queries |

### Advanced Range Queries

| # | Problem | Difficulty | Hint |
|---|---|---|---|
| 732 | [My Calendar III](https://leetcode.com/problems/my-calendar-iii/) | Hard | Coordinate compression + Segment Tree with lazy propagation or difference array map |

### Self-Check & Phase 03 Completion
If you can write a Segment Tree and a Fenwick Tree from memory, explain why $4n$ space is required, and perform bitwise index traversals, you have completed the final notebook of Phase 03. You are now ready to progress to **Phase 04 (Graph Theory)**, where we will transition from trees (which have no cycles and exactly one parent) to general graphs, exploring representations, DFS/BFS traversals, topological sorting, and shortest path algorithms.